# Faz 3 — nnUNet Segmentasyon Viewer

İnteraktif viewer: CT kesiti + GT segmentasyon maskesi (renkli) + nnUNet tahminleri.

- **Split / Sınıf / Vaka** dropdown ile filtreleme
- **Kesit slider** ile z-ekseni gezme
- **GT** ve **Tahmin** toggle butonları
- Tahmin mevcut değilse yalnızca GT gösterilir


In [1]:
!pip install -q nibabel ipywidgets

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

DATA_ROOT = Path(os.environ.get("TR_ABDOMEN_BASE", r"D:/makale-pdf/Proje/abdomen"))
PROJE     = Path(os.environ.get("TR_ABDOMEN_PROJE", r"D:/makale-pdf/Proje"))
CODE      = Path(os.environ.get("TR_ABDOMEN_CODE",  r"D:/makale-pdf/Proje/code"))
EGITIM_DIR = Path(os.environ.get("ABDOMEN_TRAIN_DIR", DATA_ROOT / "Egitim Verisi"))
YARISMA_DIR = Path(os.environ.get("ABDOMEN_TEST_DIR", DATA_ROOT / "Test Verisi"))

# os.environ["ABDOMEN_PROJECT_ROOT"] = str(PROJE)
# os.environ["ABDOMEN_DATA_ROOT"]    = str(DATA_ROOT)
# os.environ["ABDOMEN_TRAIN_DIR"]    = str(EGITIM_DIR)
# os.environ["ABDOMEN_TEST_DIR"]     = str(DATA_ROOT / "Yarışma Veri Seti")
# os.environ["ABDOMEN_BILGI_XLSX"]   = str(DATA_ROOT / "Bilgi.xlsx")
# os.environ["ABDOMEN_OUT_DIR"]      = str(PROJE / "outputs")

sys.path.insert(0, str(CODE))
from src.config import DET_DATA_DIR, CKPT_DIR, SUPER_CLASSES, DEFAULT_DET

fold = 0


YOLO veri seti: /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/det_data/fold0 → var? True
dataset.yaml : True


In [3]:
import warnings
import threading
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ipywidgets import IntSlider, Dropdown, HBox, VBox, Output, ToggleButton
from IPython.display import display
import pandas as pd
from src.splits import load_fold
from src.config import SUPER_CLASSES; warnings.filterwarnings("ignore")
try:
    import nibabel as nib
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nibabel"])
    import nibabel as nib

# ── Yollar ──────────────────────────────────────────────────────────────────
_out = Path(os.environ.get("ABDOMEN_OUT_DIR", str(PROJE / "outputs")))
NIFTI_DIR = _out / "nndet" / "nifti"
LABEL_DIR = _out / "nndet" / "nnunet_raw" / "Dataset100_Abdomen" / "labelsTr"
PRED_DIR  = _out / "nndet" / "nnunet_results" / "Dataset100_Abdomen" / "nnUNetTrainer__nnUNetPlans__3d_fullres" / f"fold_{fold}" / "val_predictions"

print(f"NIfTI  : {NIFTI_DIR}  ({'✓' if NIFTI_DIR.exists() else '✗'})")
print(f"Labels : {LABEL_DIR}  ({'✓' if LABEL_DIR.exists() else '✗'})")
print(f"Preds  : {PRED_DIR}  ({'✓' if PRED_DIR.exists() else '✗'})")
_pred_cases = set(p.stem.replace("case_", "") for p in PRED_DIR.glob("*.nii.gz")) if PRED_DIR.exists() else set()
print(f"Val prediction sayısı: {len(_pred_cases)}")

# ── nnUNet etiket renkleri ───────────────────────────────────────────────────
# nnUNet: 0=background, 1..6 = SUPER_CLASSES[0..5]
_LABEL_COLORS = [
    (0.91, 0.30, 0.24),  # kırmızı  — acute_cholecystitis
    (0.20, 0.60, 0.86),  # mavi     — kidney_ureter_stone
    (0.18, 0.80, 0.44),  # yeşil    — acute_pancreatitis
    (0.95, 0.61, 0.07),  # turuncu  — aortic_aneurysm_dissection
    (0.61, 0.35, 0.71),  # mor      — acute_appendicitis
    (0.10, 0.74, 0.61),  # camgöbeği— acute_diverticulitis
]
_OVERLAY_ALPHA = 0.50

# ── Vaka → sınıf seti (manifest üzerinden hızlı) ────────────────────────────
_manifest = pd.read_csv(_out / "splits" / "manifest.csv")
_fold_train = set(load_fold(fold, "train"))
_fold_val   = set(load_fold(fold, "val"))
_all_cases  = sorted(_fold_train | _fold_val)

def _case_super_classes(case):
    sub = _manifest[_manifest["case"] == case]["super_labels"].dropna()
    out = set()
    for v in sub:
        try: out.add(int(float(v)))
        except: pass
    return out

_case_cls_map = {c: _case_super_classes(c) for c in _all_cases}

# ── NIfTI cache ──────────────────────────────────────────────────────────────
_nifti_cache: dict = {}

def _load_nifti(case):
    if case not in _nifti_cache:
        img_p  = NIFTI_DIR / f"case_{case}_0000.nii.gz"
        lbl_p  = LABEL_DIR / f"case_{case}.nii.gz"
        pred_p = PRED_DIR  / f"case_{case}.nii.gz"
        img  = nib.load(str(img_p)).get_fdata()  if img_p.exists()  else None
        lbl  = nib.load(str(lbl_p)).get_fdata().astype(np.uint8)  if lbl_p.exists()  else None
        pred = nib.load(str(pred_p)).get_fdata().astype(np.uint8) if pred_p.exists() else None
        _nifti_cache[case] = (img, lbl, pred)
    return _nifti_cache[case]

def _hu_to_gray(arr2d, wl=40, ww=400):
    lo, hi = wl - ww / 2, wl + ww / 2
    return np.clip((arr2d.astype(np.float32) - lo) / (hi - lo), 0.0, 1.0)

def _make_overlay(mask2d):
    """Integer maske → RGBA array (0=şeffaf, 1-6=renkli)."""
    h, w = mask2d.shape
    rgba = np.zeros((h, w, 4), dtype=np.float32)
    for i, rgb in enumerate(_LABEL_COLORS):
        where = mask2d == (i + 1)
        if where.any():
            rgba[where, :3] = rgb
            rgba[where,  3] = _OVERLAY_ALPHA
    return rgba


def nnunet_viewer(fold=0):
    train_cases = set(load_fold(fold, "train"))
    val_cases   = set(load_fold(fold, "val"))
    all_cases   = sorted(train_cases | val_cases)

    split_dd = Dropdown(description="Split:", value="Tümü",
                        options=["Tümü", "Train", "Val"], layout={"width": "160px"})
    class_dd = Dropdown(description="Sınıf:", value="Tümü",
                        options=["Tümü"] + list(SUPER_CLASSES), layout={"width": "280px"})
    case_dd  = Dropdown(description="Vaka:", layout={"width": "180px"})
    slider   = IntSlider(description="Kesit:", min=0, max=1, value=0,
                         continuous_update=False, layout={"width": "500px"})
    gt_btn   = ToggleButton(value=True,  description="GT (Renk)",
                            button_style="info",   layout={"width": "130px"})
    pred_btn = ToggleButton(value=True,  description="Tahmin",
                            button_style="danger", layout={"width": "130px"},
                            disabled=(len(_pred_cases) == 0))
    info_out = Output()
    img_out  = Output()

    def filtered_cases():
        sp = split_dd.value; cn = class_dd.value
        cases = all_cases
        if sp == "Train": cases = [c for c in cases if c in train_cases]
        elif sp == "Val": cases = [c for c in cases if c in val_cases]
        if cn != "Tümü":
            ci = SUPER_CLASSES.index(cn)
            cases = [c for c in cases if ci in _case_cls_map.get(c, set())]
        return cases

    def update_cases(_):
        cases = filtered_cases()
        case_dd.options = cases
        if cases: case_dd.value = cases[0]

    def draw(z_idx, case, show_gt, show_pred):
        img, lbl, pred = _load_nifti(case)
        if img is None:
            with img_out:
                img_out.clear_output(wait=True)
                print(f"⚠️  Vaka {case}: NIfTI bulunamadı.")
            return

        n_z = img.shape[2]
        z   = min(z_idx, n_z - 1)
        ct  = _hu_to_gray(img[:, :, z])

        has_gt   = show_gt   and lbl  is not None
        has_pred = show_pred and pred is not None
        n_cols   = 1 + int(has_gt) + int(has_pred)
        fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5), facecolor="#111")
        if n_cols == 1: axes = [axes]
        col = 0

        # Ham CT
        axes[col].imshow(ct, cmap="gray", vmin=0, vmax=1)
        axes[col].set_title(f"CT  z={z+1}/{n_z}", color="white", fontsize=9)
        axes[col].axis("off"); col += 1

        # GT overlay
        if has_gt:
            axes[col].imshow(ct, cmap="gray", vmin=0, vmax=1)
            axes[col].imshow(_make_overlay(lbl[:, :, z]))
            uniq = [int(v) for v in np.unique(lbl[:, :, z]) if v > 0]
            t = ", ".join(SUPER_CLASSES[u - 1] for u in uniq) if uniq else "—"
            axes[col].set_title(f"GT: {t}", color="white", fontsize=8)
            axes[col].axis("off"); col += 1

        # Pred overlay
        if has_pred:
            axes[col].imshow(ct, cmap="gray", vmin=0, vmax=1)
            axes[col].imshow(_make_overlay(pred[:, :, z]))
            uniq = [int(v) for v in np.unique(pred[:, :, z]) if v > 0]
            t = ", ".join(SUPER_CLASSES[u - 1] for u in uniq) if uniq else "—"
            axes[col].set_title(f"Tahmin: {t}", color="white", fontsize=8)
            axes[col].axis("off")

        # Lejant
        patches = [plt.Rectangle((0, 0), 1, 1, color=_LABEL_COLORS[i], alpha=0.8,
                                  label=f"{i+1}: {SUPER_CLASSES[i]}")
                   for i in range(len(SUPER_CLASSES))]
        fig.legend(handles=patches, loc="lower center", ncol=3,
                   fontsize=7, facecolor="#222", labelcolor="white", framealpha=0.8)
        tag = "Train" if case in train_cases else "Val"
        fig.suptitle(
            f"Vaka {case}  [{tag}]  |  "
            f"Sınıflar: {', '.join(SUPER_CLASSES[c] for c in sorted(_case_cls_map.get(case, set())))}",
            color="white", fontsize=10, fontweight="bold")
        plt.tight_layout(rect=[0, 0.08, 1, 1])
        with img_out:
            img_out.clear_output(wait=True); plt.show()

    def on_case_change(_):
        case = case_dd.value
        if case is None: return
        img, lbl, pred = _load_nifti(case)
        n = img.shape[2] if img is not None else 1
        slider.max   = max(0, n - 1)
        slider.value = n // 2
        with info_out:
            info_out.clear_output()
            tag = "Train" if case in train_cases else "Val"
            gt_voxels   = int((lbl  > 0).sum()) if lbl  is not None else 0
            pred_voxels = int((pred > 0).sum()) if pred is not None else 0
            gt_cls = [SUPER_CLASSES[int(v)-1] for v in np.unique(lbl)[1:]] if lbl is not None else []
            print(f"Vaka {case}  [{tag}]  |  {n} kesit  |  "
                  f"GT voksel: {gt_voxels:,}  |  Pred voksel: {pred_voxels:,}  |  "
                  f"GT sınıflar: {', '.join(gt_cls) or '—'}")
        draw(slider.value, case, gt_btn.value, pred_btn.value)

    def on_change(_):
        if case_dd.value is not None:
            draw(slider.value, case_dd.value, gt_btn.value, pred_btn.value)

    for w in [split_dd, class_dd]: w.observe(update_cases, names="value")
    case_dd.observe(on_case_change, names="value")
    for w in [slider, gt_btn, pred_btn]: w.observe(on_change, names="value")

    display(VBox([
        HBox([split_dd, class_dd]),
        HBox([case_dd, gt_btn, pred_btn]),
        info_out, slider, img_out,
    ]))
    threading.Timer(0.3, update_cases, args=[None]).start()


NIfTI  : /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/nndet/nifti  (✓)
Labels : /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/nndet/nnunet_raw/Dataset100_Abdomen/labelsTr  (✓)
Preds  : /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/nndet/nnunet_results/Dataset100_Abdomen/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_0/val_predictions  (✓)
Val prediction sayısı: 0


In [4]:
nnunet_viewer(fold=fold)